---
# 2. SFT VIA LoRA – Mistral-7B-Instruct-v0.2
---

In [ ]:
# ---------- 1. Mount Drive & install ----------
# NOTE: Mistral-7B is not a gated model. HF login is optional.
# If you have other gated models in your workflow, add HF_TOKEN to Colab Secrets.
from google.colab import drive
drive.mount('/content/drive')

!pip uninstall -y torchao
!pip install -q transformers accelerate peft datasets requests

In [ ]:
# ---------- 2. HuggingFace login (optional - Mistral is not gated) ----------
try:
    from google.colab import userdata
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("HF login successful.")
    else:
        print("No HF_TOKEN found. Skipping login (fine for Mistral).")
except Exception:
    print("HF login skipped.")

In [ ]:
# ---------- 3. Imports ----------
import json, requests, torch
from dataclasses import dataclass
from typing import Any
 
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
 
from peft import LoraConfig, get_peft_model
from datasets import Dataset

In [ ]:
# ---------- 4. Paths ----------
# UPDATE GITHUB_RAW to point to your repo's dataset folder
GITHUB_RAW = "https://raw.githubusercontent.com/Heatbless/slm-safety-comparison/main/dataset"
SFT_FILE   = "sft_dataset.jsonl"
 
# Drive is used for saving the trained LoRA adapter
BASE_DIR   = "/content/drive/MyDrive/Colab Notebooks/slm_safety"
OUTPUT_DIR = BASE_DIR + "/mistral_sft_lora"
 
print(f"GitHub base : {GITHUB_RAW}")
print(f"Adapter out : {OUTPUT_DIR}")

In [ ]:
# ---------- 5. GitHub JSONL loader ----------
def load_jsonl_from_github(base_url, filename):
    url = f"{base_url}/{filename}"
    response = requests.get(url)
    response.raise_for_status()
    return [
        json.loads(line)
        for line in response.text.strip().splitlines()
        if line.strip()
    ]

In [ ]:
# ---------- 6. Load model & tokenizer ----------
# Training runs in full BF16 (no quantization) to maximize weight update precision.
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
 
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
 
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
# ---------- 7. LoRA config ----------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
 
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ---------- 8. Load SFT dataset from GitHub ----------
sft_data = load_jsonl_from_github(GITHUB_RAW, SFT_FILE)
dataset  = Dataset.from_list(sft_data)
print(f"Loaded {len(dataset)} examples from GitHub")
 
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]
 
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

In [ ]:
# ---------- 9. Instruction masking helpers ----------
inst_tokens = tokenizer.encode("[/INST]", add_special_tokens=False)
 
def find_subsequence(sequence, subsequence):
    for i in range(len(sequence) - len(subsequence) + 1):
        if sequence[i:i + len(subsequence)] == subsequence:
            return i
    return -1

In [ ]:
# ---------- 10. Tokenization ----------
def format_and_tokenize(example):
    full_text = (
        f"<s>[INST] {example['prompt']} [/INST] "
        f"{example['response']}</s>"
    )
 
    enc = tokenizer(full_text, truncation=True, max_length=512)
    input_ids = enc["input_ids"]
 
    inst_start = find_subsequence(input_ids, inst_tokens)
 
    if inst_start == -1:
        last_inst_idx = 0
    else:
        last_inst_idx = inst_start + len(inst_tokens) - 1
 
    labels = [-100] * (last_inst_idx + 1) + input_ids[last_inst_idx + 1:]
    enc["labels"] = labels
 
    return enc
 
tokenized_train = train_dataset.map(format_and_tokenize, remove_columns=train_dataset.column_names)
tokenized_eval  = eval_dataset.map(format_and_tokenize, remove_columns=eval_dataset.column_names)
 
print(f"Train examples: {len(tokenized_train)}")
print(f"Eval examples : {len(tokenized_eval)}")

In [ ]:
# ---------- 11. Custom collator ----------
@dataclass
class SFTCollator:
    tokenizer: Any
 
    def __call__(self, features):
        labels = [f["labels"] for f in features]
        inputs = [
            {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
            for f in features
        ]
 
        batch   = self.tokenizer.pad(inputs, padding=True, return_tensors="pt")
        max_len = batch["input_ids"].shape[1]
 
        padded_labels = [
            label + [-100] * (max_len - len(label))
            for label in labels
        ]
 
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch
 
data_collator = SFTCollator(tokenizer)

In [ ]:
# ---------- 12. Training arguments ----------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
 
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
 
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
 
    bf16=True,
    logging_steps=5,
 
    eval_strategy="steps",
    eval_steps=5,
 
    save_strategy="steps",
    save_steps=5,
    save_total_limit=2,
 
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
 
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
# ---------- 13. Trainer & train ----------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
 
trainer.train()

In [ ]:
# ---------- 14. Save adapter ----------
model.save_pretrained(OUTPUT_DIR)
print(f"SFT adapter saved to {OUTPUT_DIR}")

# Evaluation of SFT Hardened Mistral-7B v0.2

In [ ]:
# ---------- 1. Mount Drive & install ----------
from google.colab import drive
drive.mount('/content/drive')
 
!pip uninstall -y torchao
!pip install -q transformers accelerate peft datasets bitsandbytes requests

In [ ]:
# ---------- 2. Imports ----------
import json, os, time, requests, warnings
import pandas as pd
import torch
 
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

warnings.filterwarnings("ignore")

In [ ]:
# ---------- 3. Paths ----------
GITHUB_RAW  = "https://raw.githubusercontent.com/Heatbless/slm-safety-comparison/main/dataset"
 
PROMPT_FILES = [
    "general_question.jsonl",
    "lora_question.jsonl",
]

# Drive is used for loading the saved adapter and saving results
BASE_DIR    = "/content/drive/MyDrive/Colab Notebooks/slm_safety"
ADAPTER_DIR = BASE_DIR + "/mistral_sft_lora"
OUTPUT_CSV  = BASE_DIR + "/dataset/sft_eval_results.csv"
 
print(f"GitHub base : {GITHUB_RAW}")
print(f"Adapter dir : {ADAPTER_DIR}")
print(f"Output CSV  : {OUTPUT_CSV}")

In [ ]:
# ---------- 4. GitHub JSONL loader ----------
def load_jsonl_from_github(base_url, filename):
    url = f"{base_url}/{filename}"
    response = requests.get(url)
    response.raise_for_status()
    return [
        json.loads(line)
        for line in response.text.strip().splitlines()
        if line.strip()
    ]

In [ ]:
# ---------- 5. Load tokenizer ----------
base_model_id = "mistralai/Mistral-7B-Instruct-v0.2"
 
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

In [ ]:
# ---------- 6. Quantization config ----------
# NOTE: weights stored in 4-bit NF4, dequantized to BF16 for computation.
# VRAM usage reflects BF16 levels during inference due to dequantization.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# ---------- 7. Load base model + LoRA adapter ----------
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("SFT adapter loaded.")

In [ ]:
# ---------- 8. Load prompts from GitHub ----------
all_prompts = []
for fname in PROMPT_FILES:
    try:
        prompts = load_jsonl_from_github(GITHUB_RAW, fname)
        all_prompts.extend(prompts)
        print(f"Loaded {len(prompts)} prompts from {fname}")
    except Exception as e:
        print(f"Failed to load {fname}: {e}, skipping.")
 
prompt_df = pd.DataFrame(all_prompts)
print(f"\nTotal: {len(prompt_df)} prompts loaded")

In [ ]:
# ---------- 9. Generation parameters ----------
SYSTEM_PROMPTS = {"A": ""}
MAX_NEW_TOKENS = 256
TEMPERATURE    = 0.0
DO_SAMPLE      = False
SEED           = 42
 
torch.manual_seed(SEED)

In [ ]:
# ---------- 10. Inference loop ----------
results = []
 
for cond, sys_prompt in SYSTEM_PROMPTS.items():
    base_messages = []
    if sys_prompt:
        base_messages.append({"role": "system", "content": sys_prompt})
 
    for idx, row in prompt_df.iterrows():
        prompt_text = row["prompt"]
        messages    = base_messages.copy()
        messages.append({"role": "user", "content": prompt_text})
 
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
 
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                do_sample=DO_SAMPLE,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
 
        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
 
        results.append({
            "model"                  : "Mistral-SFT",
            "system_prompt_condition": cond,
            "prompt_id"              : row["prompt_id"],
            "technique"              : row["technique"],
            "difficulty"             : row["difficulty"],
            "prompt"                 : prompt_text,
            "response"               : response_text,
            "input_tokens"           : inputs["input_ids"].shape[1],
            "output_tokens"          : len(generated_ids),
        })
 
        if len(results) % 20 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            print(f"Progress: {len(results)} results saved...")

In [ ]:
# ---------- B11. Final save ----------
pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
print(f"\nEvaluation complete. {len(results)} results saved to:\n{OUTPUT_CSV}")